# ACHG-CLIP: Experiment B2 (ViT-L/14 Backbone Study) - CUB200
This notebook trains and evaluates the ACHG-CLIP model on CUB200 FSCIL using the **ViT-L/14** backbone with **Seed 42**.

In [ ]:
import torch
print('='*50)
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name    : {torch.cuda.get_device_name(0)}')
    print(f'Device VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print('='*50)

In [ ]:
!pip install -q torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers ftfy regex pyyaml "numpy<2"

import os
if not os.path.exists('/kaggle/working/FSCIL'):
    !git clone https://github.com/Siddarth021/FSCIL.git /kaggle/working/FSCIL

os.chdir('/kaggle/working/FSCIL/ACHG-CLIP')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Write latest run_cub200.py
code_cifar = "import os\nimport sys\nimport yaml\nimport time\nimport json\nimport torch\nimport torch.nn as nn\nimport argparse\nimport shutil\nfrom datetime import datetime\n\nfrom data.registry import get_data_manager\nfrom models.achg_clip import ACHGCLIP, ACHGCLIPConfig\nfrom models.clip.clip_wrapper import CLIPConfig\nfrom training.trainer import ACHGCLIPTrainer, TrainerConfig\nfrom evaluation.session_evaluator import FSCILSessionEvaluator\nimport transformers\n\nclass LoggerTee:\n    def __init__(self, filename):\n        self.terminal = sys.stdout\n        self.log = open(filename, \"a\", encoding=\"utf-8\")\n\n    def write(self, message):\n        self.terminal.write(message)\n        self.log.write(message)\n        self.log.flush()\n\n    def flush(self):\n        self.terminal.flush()\n        self.log.flush()\n        \n    def isatty(self):\n        return hasattr(self.terminal, 'isatty') and self.terminal.isatty()\n\n# CIFAR100 images are resized to 224x224 in our pipeline.\n# HF CLIP takes 224x224 raw images.\nMAX_TEXT_LEN = 77 # HF max length\nVOCAB_SIZE = 49408 # HF CLIP vocab size\n\nclass SimplePatchifier(nn.Module):\n    def __init__(self, patch_size=32):\n        super().__init__()\n        self.patch_size = patch_size\n        \n    def forward(self, images):\n        # HF CLIP accepts images directly, so we just return them for HF.\n        return images\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--short\", action=\"store_true\", help=\"Run short validation (2 base epochs, 1 inc epoch)\")\n    parser.add_argument(\"--resume_base\", type=str, default=\"\", help=\"Path to a checkpoint to resume base training from\")\n    parser.add_argument(\"--variant\", type=str, default=\"\", help=\"CLIP backbone variant (ViT-B/32, ViT-B/16, ViT-L/14)\")\n    parser.add_argument(\"--seed\", type=int, default=None, help=\"Seed for class split / data manager\")\n    parser.add_argument(\"--data_root\", type=str, default=\"./datasets\", help=\"Path to data root directory\")\n    parser.add_argument(\"--base_epochs\", type=int, default=None, help=\"Base epochs (overrides config)\")\n    parser.add_argument(\"--incremental_epochs\", type=int, default=None, help=\"Incremental epochs (overrides config)\")\n    parser.add_argument(\"--base_batch_size\", type=int, default=None, help=\"Base batch size\")\n    parser.add_argument(\"--incremental_batch_size\", type=int, default=None, help=\"Incremental batch size\")\n    args = parser.parse_args()\n\n    # 1. Generate run folder name\n    run_dir = os.path.join(\"results\", datetime.now().strftime(\"run_%d%m%Y_%H%M%S\"))\n    os.makedirs(run_dir, exist_ok=True)\n    \n    # 2. Redirect stdout/stderr to train_log.txt inside the run folder\n    sys.stdout = LoggerTee(os.path.join(run_dir, \"train_log.txt\"))\n    sys.stderr = sys.stdout\n    \n    print(f\"Starting run. Logs and checkpoints will be saved to {run_dir}\")\n    \n    # 3. Copy configuration files to the run folder for reproducibility\n    config_dest = os.path.join(run_dir, \"configs\")\n    os.makedirs(config_dest, exist_ok=True)\n    for cfg_file in [\"configs/data/cub200.yaml\", \"configs/training.yaml\", \"configs/model/clip_backbone.yaml\"]:\n        if os.path.exists(cfg_file):\n            shutil.copy(cfg_file, config_dest)\n            print(f\"Copied {cfg_file} to run directory.\")\n            \n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    \n    with open(\"configs/data/cub200.yaml\", \"r\") as f:\n        data_cfg = yaml.safe_load(f)\n    with open(\"configs/training.yaml\", \"r\") as f:\n        train_cfg = yaml.safe_load(f)\n    with open(\"configs/model/clip_backbone.yaml\", \"r\") as f:\n        model_cfg = yaml.safe_load(f)\n\n    # Apply overrides\n    if args.seed is not None:\n        if isinstance(data_cfg.get(\"seed\"), dict):\n            data_cfg[\"seed\"][\"value\"] = args.seed\n        else:\n            data_cfg[\"seed\"] = args.seed\n    if args.base_batch_size is not None:\n        data_cfg[\"base_batch_size\"][\"value\"] = args.base_batch_size\n    if args.incremental_batch_size is not None:\n        data_cfg[\"incremental_batch_size\"][\"value\"] = args.incremental_batch_size\n    if args.base_epochs is not None:\n        data_cfg[\"base_epochs\"][\"value\"] = args.base_epochs\n    if args.incremental_epochs is not None:\n        data_cfg[\"incremental_epochs\"][\"value\"] = args.incremental_epochs\n\n    variant = args.variant if args.variant else model_cfg.get(\"variant\", {}).get(\"value\", \"ViT-B/32\")\n\n    # Resolve data root path\n    data_root = args.data_root\n    if not os.path.exists(data_root):\n        if os.path.exists(\"D:/FSCIL/datasets\"):\n            data_root = \"D:/FSCIL/datasets\"\n        elif os.path.exists(\"f:/FSCIL/datasets\"):\n            data_root = \"f:/FSCIL/datasets\"\n        else:\n            os.makedirs(data_root, exist_ok=True)\n\n    # Instantiate the data manager\n    print(f\"\\nInitializing Data Manager with data_root={data_root}...\")\n    manager = get_data_manager(data_cfg, data_root=data_root, synthetic=False)\n    \n    print(f\"Dataset availability: Verified.\")\n    print(f\"Base classes (Session 0): {manager.base_classes}\")\n    print(f\"Incremental classes: {manager.incremental_classes}\")\n    print(f\"Total sessions: {manager.num_sessions}\")\n\n    # Initialize CLIPProcessor matching variant\n    hf_model_map = {\n        \"ViT-B/32\": \"openai/clip-vit-base-patch32\",\n        \"ViT-B/16\": \"openai/clip-vit-base-patch16\",\n        \"ViT-L/14\": \"openai/clip-vit-large-patch14\"\n    }\n    hf_model_name = hf_model_map.get(variant, \"openai/clip-vit-base-patch32\")\n    print(f\"\\nLoading HuggingFace CLIPProcessor for {variant} ({hf_model_name})...\")\n    processor = transformers.CLIPProcessor.from_pretrained(hf_model_name)\n\n    # Get class names\n    if hasattr(manager.train_dataset, 'dataset') and hasattr(manager.train_dataset.dataset, 'classes'):\n        class_names = manager.train_dataset.dataset.classes\n    elif hasattr(manager.train_dataset, 'classes'):\n        class_names = manager.train_dataset.classes\n    else:\n        # Fallback if classes attribute missing\n        class_names = [f\"class {i}\" for i in range(100)]\n    \n    # Pre-tokenize all 100 classes\n    text_prompts = [f\"a photo of a {name}\" for name in class_names]\n    text_inputs = processor(text=text_prompts, return_tensors=\"pt\", padding=\"max_length\", max_length=77, truncation=True)\n    global_tokens = text_inputs.input_ids.to(device)\n\n    print(f\"Selected CLIP Variant: {variant}\")\n\n    if variant in [\"ViT-B/32\", \"ViT-B/16\"]:\n        d_model = 768\n        d_e = 512\n        d_k = 64\n        num_heads = 12\n        num_layers = 12\n        ffn_hidden_dim = 3072\n    elif variant == \"ViT-L/14\":\n        d_model = 1024\n        d_e = 768\n        d_k = 64\n        num_heads = 16\n        num_layers = 24\n        ffn_hidden_dim = 4096\n    else:\n        raise ValueError(f\"Unknown variant: {variant}\")\n\n    # Build model Config\n    clip_cfg = CLIPConfig(\n        variant=variant,\n        d_model=d_model,\n        d_e=d_e,\n        d_k=d_k,\n        num_heads=num_heads,\n        num_layers=num_layers,\n        vocab_size=VOCAB_SIZE,\n        max_text_len=MAX_TEXT_LEN,\n        patch_dim=1, # not used by HF, but must be > 0 for validation\n        max_patches=1, # not used by HF, but must be > 0 for validation\n        ffn_hidden_dim=ffn_hidden_dim,\n        dropout=0.0\n    )\n\n    achg_cfg = ACHGCLIPConfig(\n        clip=clip_cfg,\n        num_layers=num_layers,\n        prompt_dim=d_model,\n        node_feature_dim=d_model,\n        mlp_bridge_hidden_dim=256,\n        gin_num_layers=4,\n        gin_hidden_dim=16,\n        acga_latent_dim=64,\n        hgnec_compressed_dim=8,\n        hgnec_restored_dim=16, # restored to gin_hidden_dim\n    )\n\n    print(\"Building model...\")\n    model = ACHGCLIP(achg_cfg)\n    \n    print(\"Setting up TrainerConfig...\")\n    t_cfg = TrainerConfig(\n        lr=train_cfg.get(\"learning_rate\", {}).get(\"value\", 0.000325),\n        weight_decay=train_cfg.get(\"weight_decay\", {}).get(\"value\", 0.001),\n        gradient_accumulation_steps=train_cfg.get(\"gradient_accumulation_steps\", {}).get(\"value\", 3),\n        gradient_clip_max_norm=train_cfg.get(\"gradient_clip_max_norm\", {}).get(\"value\", 4.0),\n    )\n    \n    print(\"Setting up Trainer...\")\n    trainer = ACHGCLIPTrainer(model, t_cfg, device)\n    \n    print(\"Setting up Patchifier...\")\n    patchifier = SimplePatchifier().to(device)\n    \n    # Evaluation Wrapper Setup\n    def get_text_tokens(labels, device):\n        return global_tokens[labels]\n        \n    class WrapperModel(nn.Module):\n        def __init__(self, core_model, patchifier):\n            super().__init__()\n            self.core_model = core_model\n            self.patchifier = patchifier\n            \n        def forward(self, images, labels=None):\n            patches = self.patchifier(images)\n            num_classes = 100\n            tokens = get_text_tokens(torch.arange(num_classes, device=images.device), images.device)\n            out = self.core_model(patches, tokens, dt=0.01)\n            \n            h_v = nn.functional.normalize(out.h_vision, dim=-1)\n            h_t = nn.functional.normalize(out.h_text, dim=-1)\n            \n            logit_scale = 100.0\n            logits = logit_scale * h_v @ h_t.T\n            return logits\n\n    eval_wrapper = WrapperModel(model, patchifier).to(device)\n    evaluator = FSCILSessionEvaluator(\n        model=eval_wrapper,\n        device=device,\n        data_manager=manager,\n    )\n\n    # ---------------------------------------------------------\n    # Base Training (Session 0)\n    # ---------------------------------------------------------\n    base_epochs = 2 if args.short else data_cfg.get(\"base_epochs\", {}).get(\"value\", 3)\n    incremental_epochs = 1 if args.short else data_cfg.get(\"incremental_epochs\", {}).get(\"value\", 5)\n    \n    print(\"\\n==================================================\")\n    print(f\"SESSION 0 (BASE TRAINING) - {base_epochs} epochs\")\n    print(\"==================================================\")\n    session_0 = manager.get_session(0)\n    \n    if args.resume_base:\n        print(f\"Resuming base training from {args.resume_base}\")\n        trainer.load_checkpoint(args.resume_base)\n    \n    best_acc = 0.0\n    start_time = time.time()\n    \n    for epoch in range(base_epochs):\n        print(f\"\\nEpoch {epoch+1}/{base_epochs}\")\n        trainer.train_mode()\n        \n        batch_start_time = time.time()\n        for batch_idx, (images, targets) in enumerate(session_0.train_loader):\n            images = images.to(device)\n            targets = targets.to(device)\n            \n            tokens = global_tokens\n            patches = patchifier(images)\n            loss_dict = trainer.train_step(patches, tokens, targets, dt=0.01)\n            \n            if batch_idx % 20 == 0:\n                elapsed = time.time() - batch_start_time\n                ms_per_batch = (elapsed / 20 * 1000) if batch_idx > 0 else 0\n                print(f\"  Batch {batch_idx}: Total Loss = {loss_dict['L_total']:.4f} | {ms_per_batch:.1f} ms/batch\")\n                batch_start_time = time.time()\n                \n        # Evaluate after epoch\n        print(\"  Evaluating base session accuracy...\")\n        metrics_0 = evaluator.evaluate_session(0)\n        acc = metrics_0['accuracy']\n        print(f\"  Epoch {epoch+1} Accuracy: {acc:.4f}\")\n        \n        # 4. Save Checkpoints\n        trainer.save_checkpoint(os.path.join(run_dir, \"latest_checkpoint.pt\"), seed=42)\n        \n        if acc > best_acc:\n            best_acc = acc\n            trainer.save_checkpoint(os.path.join(run_dir, \"best_checkpoint.pt\"), seed=42)\n            print(f\"  --> New best accuracy! Saved best_checkpoint.pt\")\n            \n        if (epoch + 1) % 5 == 0:\n            trainer.save_checkpoint(os.path.join(run_dir, f\"epoch_{epoch+1}.pt\"), seed=42)\n\n    end_time = time.time()\n    print(f\"\\nSession 0 Training completed in {end_time - start_time:.2f} seconds.\")\n    \n    accuracies = {0: best_acc}\n    \n    # ---------------------------------------------------------\n    # Incremental Sessions (1-8)\n    # ---------------------------------------------------------\n    print(\"\\n--- Starting Incremental Training & Evaluation ---\")\n    \n    for session_idx in range(1, manager.num_sessions):\n        print(f\"\\n==================================================\")\n        print(f\"SESSION {session_idx}\")\n        print(f\"==================================================\")\n        \n        # Load previous session checkpoint or best base checkpoint\n        prev_session_ckpt = os.path.join(run_dir, f\"session_{session_idx-1}.pt\")\n        if session_idx == 1:\n            prev_session_ckpt = os.path.join(run_dir, \"best_checkpoint.pt\")\n            \n        print(f\"Loading checkpoint from: {prev_session_ckpt}\")\n        trainer.load_checkpoint(prev_session_ckpt)\n        \n        session_data = manager.get_session(session_idx)\n        \n        print(\"Training...\")\n        trainer.train_mode()\n        \n        for epoch in range(incremental_epochs):\n            batch_start_time = time.time()\n            for batch_idx, (images, targets) in enumerate(session_data.train_loader):\n                images = images.to(device)\n                targets = targets.to(device)\n                \n                tokens = global_tokens\n                patches = patchifier(images)\n                \n                loss_dict = trainer.train_step(patches, tokens, targets, dt=0.01)\n                \n                if batch_idx % 20 == 0:\n                    elapsed = time.time() - batch_start_time\n                    ms_per_batch = (elapsed / 20 * 1000) if batch_idx > 0 else 0\n                    print(f\"  Batch {batch_idx}: Total Loss = {loss_dict['L_total']:.4f} | {ms_per_batch:.1f} ms/batch\")\n                    batch_start_time = time.time()\n        \n        curr_session_ckpt = os.path.join(run_dir, f\"session_{session_idx}.pt\")\n        trainer.save_checkpoint(curr_session_ckpt, seed=42)\n        print(f\"Saved checkpoint to {curr_session_ckpt}\")\n        \n        print(f\"Evaluating cumulative classes...\")\n        metrics = evaluator.evaluate_session(session_idx)\n        print(f\"Cumulative accuracy: {metrics['accuracy']:.4f}\")\n        accuracies[session_idx] = metrics['accuracy']\n        \n    print(\"\\nValidation Complete.\")\n    \n    # Calculate final metrics requested by the user\n    a_base = accuracies[0]\n    a_last = accuracies[manager.num_sessions - 1]\n    pd = a_base - a_last\n    mean_acc = sum(accuracies.values()) / len(accuracies)\n    \n    total_runtime = time.time() - start_time # Rough approximation for full script runtime\n    \n    final_metrics = {\n        \"Base accuracy (A_base)\": a_base,\n        \"Accuracy after every incremental session\": accuracies,\n        \"A_last\": a_last,\n        \"Mean accuracy\": mean_acc,\n        \"Performance Drop (PD)\": pd,\n        \"Runtime\": total_runtime,\n        \"Exact configuration used\": {\n            \"Data Config\": data_cfg,\n            \"Train Config\": train_cfg,\n        }\n    }\n    \n    print(\"\\n\" + \"=\"*60)\n    print(\"--- FINAL EXPERIMENTAL RESULTS ---\")\n    print(\"=\"*60)\n    print(f\"Backbone Variant : {variant}\")\n    print(f\"Random Seed      : {data_cfg.get('seed', {}).get('value', 42) if isinstance(data_cfg.get('seed'), dict) else data_cfg.get('seed', 42)}\")\n    print(f\"Base Accuracy    : {a_base*100:.2f}%\")\n    print(f\"Final Accuracy   : {a_last*100:.2f}%\")\n    print(f\"Performance Drop : {pd*100:.2f}%\")\n    print(f\"Mean Accuracy    : {mean_acc*100:.2f}%\")\n    print(f\"Total Runtime    : {total_runtime:.1f}s\")\n    print(\"\\nSession Accuracies:\")\n    for s_idx in sorted(accuracies.keys()):\n        print(f\"  Session {s_idx}: {accuracies[s_idx]*100:.2f}%\")\n    print(\"=\"*60 + \"\\n\")\n    \n    # Write to a summary log in results directory\n    with open(os.path.join(run_dir, \"eval_summary.json\"), \"w\") as f:\n        json.dump(final_metrics, f, indent=4)\n\nif __name__ == \"__main__\":\n    main()\n"
with open('run_cub200.py', 'w', encoding='utf-8') as f:
    f.write(code_cifar)
print('Updated run_cub200.py')

# Write latest run_experiments.py
code_exp = "import yaml\nimport os\nimport subprocess\nimport shutil\n\ndef modify_yaml(filepath, key, value):\n    with open(filepath, \"r\") as f:\n        cfg = yaml.safe_load(f)\n    if key in cfg:\n        if isinstance(cfg[key], dict) and \"value\" in cfg[key]:\n            cfg[key][\"value\"] = value\n        else:\n            cfg[key] = value\n    else:\n        cfg[key] = {\"value\": value}\n    with open(filepath, \"w\") as f:\n        yaml.dump(cfg, f)\n\ndef run_experiment(name, data_seed, backbone_variant):\n    print(f\"\\n==============================================\")\n    print(f\"Starting Experiment: {name}\")\n    print(f\"==============================================\")\n    \n    # 1. Modify configs\n    modify_yaml(\"configs/data/cifar100.yaml\", \"seed\", data_seed)\n    modify_yaml(\"configs/model/clip_backbone.yaml\", \"variant\", backbone_variant)\n    \n    # 2. Run the training script\n    subprocess.run([\"python\", \"run_cifar100.py\"], check=True)\n    \n    # 3. Find the most recently created run directory in results/\n    results_dir = \"results\"\n    subdirs = [os.path.join(results_dir, d) for d in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, d)) and d.startswith(\"run_\")]\n    if subdirs:\n        latest_run = max(subdirs, key=os.path.getmtime)\n        \n        # 4. Rename it to the experiment name for easy tracking\n        new_dir = os.path.join(results_dir, name)\n        if os.path.exists(new_dir):\n            shutil.rmtree(new_dir)\n        os.rename(latest_run, new_dir)\n        print(f\"Experiment {name} saved to {new_dir}\\n\")\n\nif __name__ == \"__main__\":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--experiment\", type=str, default=\"all\", choices=[\"all\", \"SplitStudy\", \"Backbone_CLIP_B\", \"Backbone_CLIP_C\"], help=\"Experiment to run\")\n    args = parser.parse_args()\n\n    # Baseline configs to restore later\n    original_seed = 42\n    original_variant = \"ViT-B/32\"\n    \n    try:\n        if args.experiment in [\"all\", \"SplitStudy\"]:\n            # Experiment A: Split B (Seed 1993, ViT-B/32)\n            run_experiment(\"SplitStudy\", 1993, \"ViT-B/32\")\n            \n        if args.experiment in [\"all\", \"Backbone_CLIP_B\"]:\n            # Experiment B1: Backbone CLIP-B (Seed 42, ViT-B/16)\n            run_experiment(\"Backbone_CLIP_B\", 42, \"ViT-B/16\")\n            \n        if args.experiment in [\"all\", \"Backbone_CLIP_C\"]:\n            # Experiment B2: Backbone CLIP-C (Seed 42, ViT-L/14)\n            run_experiment(\"Backbone_CLIP_C\", 42, \"ViT-L/14\")\n            \n    finally:\n        # Always restore baseline configs\n        modify_yaml(\"configs/data/cifar100.yaml\", \"seed\", original_seed)\n        modify_yaml(\"configs/model/clip_backbone.yaml\", \"variant\", original_variant)\n        print(\"Restored baseline configs.\")\n"
with open('run_experiments.py', 'w', encoding='utf-8') as f:
    f.write(code_exp)
print('Updated run_experiments.py')

In [ ]:
import subprocess
import sys

print('Starting Experiment B2 (ViT-L/14, Seed 42)...')
cmd = [sys.executable, 'run_cub200.py', '--variant', 'ViT-L/14', '--seed', '42', '--data_root', './datasets']

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {process.returncode}')

In [ ]:
import os
import json
import shutil

results_dir = 'results'
subdirs = [os.path.join(results_dir, d) for d in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, d)) and d.startswith('run_')]
if subdirs:
    latest_run = max(subdirs, key=os.path.getmtime)
    print(f'Latest run folder: {latest_run}')
    summary_file = os.path.join(latest_run, 'eval_summary.json')
    if os.path.exists(summary_file):
        with open(summary_file, 'r') as f:
            summary = json.load(f)
        print('\n' + '='*60)
        print('FINAL EVALUATION SUMMARY')
        print('='*60)
        print(json.dumps(summary, indent=2))
        shutil.copy(summary_file, '/kaggle/working/eval_summary.json')
    
    # Copy results artifacts to /kaggle/working
    dest = '/kaggle/working/Backbone_CLIP_C_ViTL14_CUB200'
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(latest_run, dest)
    print(f'\nArtifacts saved to {dest}')